# Download images from Planet.com orders

**Purpose:** Downloads assets from Planet
[Orders API](https://developers.planet.com/apis/orders/) orders into `/kaggle/working`.

**Synced with `create_planet_orders.ipynb`:** Set the same `start_idx` / `end_idx`
here as you used when creating orders — only the matching orders (named
`incident_<ID>_after`) will be downloaded. Leave both as `0` to download **all**
successful orders (original behaviour).

This notebook runs on **Kaggle**.

## Setup (one time)
1. Get your Planet API key from https://www.planet.com/account/#/ (or your Planet account settings).
2. In the Kaggle notebook editor: **Add-ons -> Secrets** -> add a secret named
   `planet_api_key` with your API key as the value, and attach it to this notebook.

## What it does
1. Authenticates against the Planet Orders API using HTTP basic auth (API key as username).
2. Lists **all** orders (following pagination), keeping only orders in `success` or `partial` state.
3. Filters to the index range you specify (syncs with `create_planet_orders.ipynb`).
4. For each matching order, reads its `results` links and downloads every file, preserving a
   `planet_orders/<order_name>_<order_id>/` folder layout.
5. Skips files that were already downloaded (idempotent / resumable), and retries on
   transient network / rate-limit errors.


In [ ]:
# --------------------------------------------------------------------
# Imports & configuration
# --------------------------------------------------------------------
import os
import re
import pandas as pd
import requests
from requests.adapters import HTTPAdapter, Retry
from concurrent.futures import ThreadPoolExecutor, as_completed
from kaggle_secrets import UserSecretsClient

# Planet Orders API base URL
ORDERS_URL = "https://api.planet.com/compute/ops/orders/v2"

# Where to store the downloaded imagery on Kaggle
DOWNLOAD_DIR = "/kaggle/working/planet_orders"

# Only download orders in these states (successful orders have downloadable results)
WANTED_STATES = {"success", "partial"}

# --- Sync with create_planet_orders.ipynb ---
# Set the same CSV path and index range you used when creating orders.
# Leave start_idx=end_idx=0 to download ALL successful orders.
input_csv = "/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv"
start_idx = 0
end_idx   = 0
# ---------------------------------------------

# Concurrency / robustness knobs
MAX_WORKERS = 4          # parallel file downloads (keep modest to avoid 429s)
CHUNK_SIZE = 1 << 20     # 1 MiB streaming chunks
REQUEST_TIMEOUT = 300    # seconds per file request

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

In [ ]:
# --------------------------------------------------------------------
# Authenticate: Planet
# --------------------------------------------------------------------
user_secrets = UserSecretsClient()
PLANET_API_KEY = user_secrets.get_secret("planet_api_key")

# A shared session with retry/backoff for transient errors and rate limits.
def make_session():
    s = requests.Session()
    s.auth = (PLANET_API_KEY, "")  # Planet uses the API key as the username, blank password
    retries = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
        respect_retry_after_header=True,
    )
    s.mount("https://", HTTPAdapter(max_retries=retries, pool_maxsize=MAX_WORKERS * 2))
    return s

session = make_session()

# Sanity check the key works
r = session.get(ORDERS_URL, timeout=60)
r.raise_for_status()
print("Planet authentication OK.")

In [ ]:
# --------------------------------------------------------------------
# List all orders (follows pagination via _links._next)
# --------------------------------------------------------------------
def list_orders(session):
    orders = []
    url = ORDERS_URL
    page = 0
    while url:
        resp = session.get(url, timeout=120)
        resp.raise_for_status()
        data = resp.json()
        batch = data.get("orders", [])
        orders.extend(batch)
        page += 1
        print(f"Page {page}: fetched {len(batch)} orders (total {len(orders)})")
        url = data.get("_links", {}).get("_next")
    return orders

all_orders = list_orders(session)
print(f"\nTotal orders found: {len(all_orders)}")

# Summarise states
from collections import Counter
states = Counter(o.get("state") for o in all_orders)
print("Order states:", dict(states))

downloadable = [o for o in all_orders if o.get("state") in WANTED_STATES]
print(f"Downloadable orders ({'/'.join(sorted(WANTED_STATES))}): {len(downloadable)}")

# --------------------------------------------------------------------
# Filter to index range (sync with create_planet_orders.ipynb)
# --------------------------------------------------------------------
if start_idx == 0 and end_idx == 0:
    wanted_ids = None   # download ALL
    print("Index range not set (start_idx=end_idx=0) — downloading ALL orders.")
else:
    df = pd.read_csv(input_csv)
    df = df.iloc[start_idx:end_idx]
    wanted_ids = set(df['id'].astype(int).tolist())
    print(f"Filtering to {len(wanted_ids)} incidents from CSV rows {start_idx}:{end_idx}.")

    pattern = re.compile(r'^incident_(\d+)_after$')
    before = len(downloadable)
    downloadable = [
        o for o in downloadable
        if (m := pattern.match(o.get("name", "")))
        and int(m.group(1)) in wanted_ids
    ]
    print(f"Filtered from {before} to {len(downloadable)} matching orders.")

In [ ]:
# --------------------------------------------------------------------
# Helpers: build the list of files to download
# --------------------------------------------------------------------
def safe_name(name):
    """Make a string safe to use as a folder/file name component."""
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(name)).strip("_") or "unnamed"

def get_order_results(session, order):
    """Return the list of downloadable result items for an order.

    Some orders embed results in the list response; others require fetching
    the individual order detail. This handles both.
    """
    results = order.get("_links", {}).get("results")
    if results:
        return results
    # Fall back to the order detail endpoint
    order_id = order.get("id")
    detail = session.get(f"{ORDERS_URL}/{order_id}", timeout=120)
    detail.raise_for_status()
    return detail.json().get("_links", {}).get("results", []) or []

def plan_downloads(session, orders):
    """Return a list of (local_path, url) tuples for every result file."""
    tasks = []
    for order in orders:
        order_id = order.get("id", "no-id")
        order_name = order.get("name", "order")
        folder = os.path.join(DOWNLOAD_DIR, f"{safe_name(order_name)}_{safe_name(order_id)}")
        results = get_order_results(session, order)
        for item in results:
            url = item.get("location")
            if not url:
                continue
            # 'name' usually looks like '<order_id>/<file path>' -> keep the tail
            rel = item.get("name") or url.split("?")[0].split("/")[-1]
            rel = rel.split("/", 1)[-1] if "/" in rel else rel
            rel = rel.lstrip("/")
            local_path = os.path.join(folder, *[safe_name(p) for p in rel.split("/")])
            tasks.append((local_path, url))
    return tasks

download_tasks = plan_downloads(session, downloadable)
print(f"Total files to download: {len(download_tasks)}")

In [ ]:
# --------------------------------------------------------------------
# Download all files (parallel, resumable, with retry)
# --------------------------------------------------------------------
def download_one(task):
    local_path, url = task
    # Skip if already downloaded (non-empty file present)
    if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        return ("skip", local_path)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    tmp_path = local_path + ".part"
    try:
        with session.get(url, stream=True, timeout=REQUEST_TIMEOUT) as r:
            r.raise_for_status()
            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                    if chunk:
                        f.write(chunk)
        os.replace(tmp_path, local_path)
        return ("ok", local_path)
    except Exception as e:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        return ("error", f"{local_path}: {e}")

ok = skipped = failed = 0
errors = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(download_one, t): t for t in download_tasks}
    for i, fut in enumerate(as_completed(futures), 1):
        status, info = fut.result()
        if status == "ok":
            ok += 1
        elif status == "skip":
            skipped += 1
        else:
            failed += 1
            errors.append(info)
        if i % 25 == 0 or i == len(download_tasks):
            print(f"[{i}/{len(download_tasks)}] ok={ok} skipped={skipped} failed={failed}")

print(f"\nDone. Downloaded={ok}, already-present={skipped}, failed={failed}")
if errors:
    print("\nFirst errors:")
    for e in errors[:10]:
        print(" -", e)

In [ ]:
# --------------------------------------------------------------------
# Summary of what was downloaded
# --------------------------------------------------------------------
total_files = 0
total_bytes = 0
for root, _, files in os.walk(DOWNLOAD_DIR):
    for fn in files:
        if fn.endswith(".part"):
            continue
        total_files += 1
        total_bytes += os.path.getsize(os.path.join(root, fn))

print(f"Files on disk: {total_files}")
print(f"Total size:    {total_bytes / (1024**3):.2f} GiB")
print(f"Location:      {DOWNLOAD_DIR}")